# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Longer Content Ranks Better (Myth #4, Word Count Buckets)

**The Claim:**
The paper shows a "visibility gradient" where longer content (especially 5,000+ words) seems to earn way more impressions (8.8K) than shorter content.

**The Methodology Question:**
The paper does hedge by saying word count isn't a "magic threshold," but it still shows a direct comparison between word count and average impressions. The thing is, the data isn't a smooth curve — there's a dip in performance for mid-length pages (2K-3.5K words underperforming 1K-1.5K words) before a massive spike at 5K+ words.

So the real question is: does this word-count comparison control for content intent or topic complexity? Finding #7 in the same paper says intent and competition matter, and that depth is most useful "when a topic genuinely needs it." If the 5K+ bucket is mostly filled with comprehensive, informational pillar pages (which naturally target broader topics with high search volume), while the 2K-3.5K bucket contains narrow commercial pages, then the comparison is confounded. Are 5,000+ words causing the higher impressions, or is "5,000+ words" just a proxy for broad-topic content that independently attracts more traffic?

**Why this matters (and what would resolve it):**
If word count is just a proxy for intent, telling a writer to expand a 2,500-word commercial page to 5,000 words won't bring in 8.8K impressions — it'll just result in bloated content. This could be resolved if the paper broke down the 5K+ bucket by content_type or main_intent — if impressions stay high even within a single intent category (e.g., only commercial pages, controlling for length), that would support word count as a real driver. If the 5K+ bucket is dominated by one intent type, that supports the confound explanation instead.

### Finding 2: The 71% Growth Prediction Claim

**The Claim:**
The paper claims its Logistic Regression model can predict if a page will grow or decline with 71% accuracy on a holdout set (using an 80/20 split).

**The Methodology Questions:**
1. **The Split Design:** The paper mentions 71% holdout accuracy on an 80/20 split, but it doesn't say if that split was done row-by-row across all 57 brands, or if whole brands were held out. If it was a random row split, the model could just be memorizing patterns from brands it already saw in training. If whole brands were held out, then it actually learned something that generalizes. Without knowing which one happened, 71% could mean strong generalization or just memorizing — and those are very different claims.
2. **The Missing Base Rate:** Looking at the "Growth Prediction Coefficients" chart, there's no mention of the natural base rate of growing vs. declining pages in the dataset. If the data is imbalanced — say, 68% of pages are declining — then a model that just guesses "decline" every time would hit 68% accuracy. Without the base rate, we can't tell if 71% is a strong predictive signal or basically a coin flip above random guessing.

**Why this matters:**
These aren't nitpicks — they change whether the model is actually useful. In my own Week 5 work, I saw firsthand how much the split method matters: a random row split gave me 98% Precision@50 and 3,100+ captured clicks, while a grouped-by-client split on the exact same data dropped to 96% precision and only 1,274 clicks. The precision difference looks small, but the captured-clicks gap is massive — the random split was hallucinating business value by memorizing client patterns. That's why knowing whether the paper grouped by brand or split randomly isn't optional information.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import duckdb
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# 1. Load the data using DuckDB
con = duckdb.connect()
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT_DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

march_query = f"""
    SELECT content_hash_id, client_hash_id, 
           SUM(gsc_impressions) AS total_impressions, SUM(gsc_clicks) AS total_clicks,
           AVG(gsc_avg_position) AS average_position, SUM(ga4_engaged_sessions) AS total_engaged_sessions
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND client_has_ga4 IS TRUE AND client_has_gsc IS TRUE
    GROUP BY content_hash_id, client_hash_id
"""
april_query = f"""
    SELECT content_hash_id, client_hash_id, 
           SUM(gsc_impressions) AS total_impressions_april, SUM(gsc_clicks) AS total_clicks_april,
           AVG(gsc_avg_position) AS average_position_april
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
      AND client_has_ga4 IS TRUE AND client_has_gsc IS TRUE
    GROUP BY content_hash_id, client_hash_id
"""
print("Querying Hugging Face data... (this takes ~1-2 mins)")
merged_df = pd.merge(con.sql(march_query).df(), con.sql(april_query).df(), how="inner", on=["client_hash_id", "content_hash_id"])
eligible_df = merged_df[(merged_df['total_impressions'] >= 500) & (merged_df['total_impressions_april'] >= 500)].copy()

# 2. Target Creation
eligible_df['ctr_april'] = np.where(eligible_df['total_impressions_april'] > 0, eligible_df['total_clicks_april'] / eligible_df['total_impressions_april'], np.nan)
bins = [-float("inf"), 3, 10, 20, 50, float("inf")]
labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]
eligible_df["position_tier_april"] = pd.cut(eligible_df["average_position_april"], bins=bins, labels=labels).cat.add_categories(["no_data"]).fillna("no_data")
eligible_df["tier_avg_ctr_april"] = eligible_df["position_tier_april"].map(eligible_df.groupby("position_tier_april", observed=False)["ctr_april"].mean())

eligible_df['missed_clicks_april'] = eligible_df['total_impressions_april'] * (eligible_df['tier_avg_ctr_april'] - eligible_df['ctr_april']).clip(lower=0)
eligible_df['target_log_missed_clicks'] = np.log1p(eligible_df['missed_clicks_april'])

# 3. Features
eligible_df['ctr_march'] = np.where(eligible_df['total_impressions'] > 0, eligible_df['total_clicks'] / eligible_df['total_impressions'], 0.0)
for col in ['total_impressions', 'total_clicks', 'total_engaged_sessions']:
    eligible_df[f'log_{col}_march'] = np.log1p(eligible_df[col])
eligible_df['average_position_march'] = eligible_df['average_position']

X = eligible_df[['log_total_impressions_march', 'log_total_clicks_march', 'log_total_engaged_sessions_march', 'ctr_march', 'average_position_march']]
y = eligible_df['target_log_missed_clicks']

# 4. The "Cheating" Random Split
X_train_rand, X_test_rand, y_train_rand, y_test_rand, idx_train_rand, idx_test_rand = train_test_split(
    X, y, eligible_df.index, test_size=0.25, random_state=42
)
model_rand = LinearRegression().fit(X_train_rand, y_train_rand)
res_rand = eligible_df.loc[idx_test_rand].copy()
res_rand['pred'] = np.expm1(model_rand.predict(X_test_rand))
top50_rand = res_rand.sort_values(by='pred', ascending=False).head(50)
top50_rand['is_opp'] = top50_rand['ctr_april'] < (0.5 * top50_rand['tier_avg_ctr_april'])
print(f"Random Split (Cheating)   - Precision@50: {top50_rand['is_opp'].mean()*100:.1f}%, Captured Clicks: {top50_rand['missed_clicks_april'].sum():,.0f}")

# 5. The "Honest" Grouped Split
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, groups=eligible_df['client_hash_id']))
model_grp = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
res_grp = eligible_df.iloc[test_idx].copy()
res_grp['pred'] = np.expm1(model_grp.predict(X.iloc[test_idx]))
top50_grp = res_grp.sort_values(by='pred', ascending=False).head(50)
top50_grp['is_opp'] = top50_grp['ctr_april'] < (0.5 * top50_grp['tier_avg_ctr_april'])
print(f"Grouped Split (Honest)    - Precision@50: {top50_grp['is_opp'].mean()*100:.1f}%, Captured Clicks: {top50_grp['missed_clicks_april'].sum():,.0f}")


Querying Hugging Face data... (this takes ~1-2 mins)
Random Split (Cheating)   - Precision@50: 98.0%, Captured Clicks: 3,132
Grouped Split (Honest)    - Precision@50: 96.0%, Captured Clicks: 1,274


### Why the Random Split is Inflated (Client Leakage)

The random split creates the illusion of a perfectly accurate model (98.0% Precision and over 3,000 captured clicks). But this happens because a standard `train_test_split` scrambles the rows and randomly distributes pages from the *same client* into both the training set and the test set.
leakage specifically inflates magnitude-sensitive metrics more than classification-style metrics, because memorizing a client's typical scale is a different kind of cheating than memorizing a client's typical opportunity/non-opportunity split

The `GroupShuffleSplit` (our honest approach) draws the line *at the client level*. The test set consists entirely of brands the model has never seen before. The drop to 96.0% precision and 1,274 clicks shows what the model actually learned about the underlying reality of search, proving exactly why validating on randomly split data is dangerous for a portfolio model.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Category 1: Features derived from the target itself**
Our target, `target_log_missed_clicks`, is constructed from April's impressions and clicks. Our five features (`log_impressions_march`, `log_clicks_march`, `log_engaged_sessions_march`, `ctr_march`, `average_position_march`) share the exact same base metrics, but they are strictly isolated to March. Because our features rely exclusively on historical data that was finalized before April began, it is physically impossible for April's future outcome to back-propagate into March's metrics. The strict time-separation guarantees no target leakage.

**Category 2: Future information**
Our simulated decision point for this model is April 1st—the exact moment we generate the prioritized list of pages for the SEO team to optimize. The absolute newest date that any of our five features could have been computed is March 31st at 11:59 PM. Since every piece of information the model uses is finalized and known prior to the April 1st decision point, there is no forward-leaking future information.

**Category 3: Information from a system that already made choices**
Our feature list is strictly limited to `['log_impressions_march', 'log_clicks_march', 'log_engaged_sessions_march', 'ctr_march', 'average_position_march']`. Crucially, `missed_clicks_march`—the baseline's own ranking score—is completely excluded. This forces the model to learn the underlying patterns from raw signals rather than piggybacking on the baseline heuristic we already designed.

**Proof of Instrument (The Poison Test)**
To confirm the evaluation pipeline actually detects leakage rather than silently passing it through, a deliberately "poisoned" feature was injected: `leaky_test_feature = missed_clicks_april * 0.9` — 90% of the true answer, given directly to the model.

Precision@50 barely moved (96.0% → 98.0%), because the clean model was already near the metric's ceiling — with only 50 pages evaluated on a binary condition, there was little room left to climb. This shows Precision@50 is a weak instrument for detecting this kind of leakage once a model is already performing well.

Total captured clicks told the real story: clean model captured 1,274 clicks (46% of the 2,938-click theoretical maximum for this test set); the poisoned model captured 2,651 clicks (90% of the theoretical maximum) — a 108% increase, closely tracking the 0.9 scaling factor deliberately built into the poison feature. This confirms the evaluation pipeline correctly rewards a model that has direct access to the answer, and that total captured clicks, not Precision@50, is the more sensitive instrument for detecting this type of leakage in a model that's already performing reasonably well.

Conclusion: the clean Week 5 model's 96% Precision@50 / 1,274 captured clicks was not an artifact of leakage — it reflects genuine (if limited) generalization, well below what a leaked model achieves on the same test set.

*(Limitation Note: We tested the poison feature using the grouped split, not both splits — this is sufficient for this proof since the grouped split is our honest benchmark.)*

In [3]:
# POISON TEST: Deliberately inject 90% of the actual answer as a feature
eligible_df['leaky_test_feature'] = eligible_df['missed_clicks_april'] * 0.9

# Retrain with the leaky feature
X_poison = eligible_df[['log_total_impressions_march', 'log_total_clicks_march', 
                        'log_total_engaged_sessions_march', 'ctr_march', 
                        'average_position_march', 'leaky_test_feature']]
y = eligible_df['target_log_missed_clicks']

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X_poison, y, groups=eligible_df['client_hash_id']))
model_poison = LinearRegression().fit(X_poison.iloc[train_idx], y.iloc[train_idx])
res_poison = eligible_df.iloc[test_idx].copy()
res_poison['pred'] = np.expm1(model_poison.predict(X_poison.iloc[test_idx]))

top50_poison = res_poison.sort_values(by='pred', ascending=False).head(50)
top50_poison['is_opp'] = top50_poison['ctr_april'] < (0.5 * top50_poison['tier_avg_ctr_april'])

precision = top50_poison['is_opp'].mean() * 100
captured_clicks = top50_poison['missed_clicks_april'].sum()
print(f"POISON TEST - Precision@50: {precision:.1f}%, Captured Clicks: {captured_clicks:,.0f}")

POISON TEST - Precision@50: 98.0%


In [4]:
top50_poison['missed_clicks_april'].sum()

np.float64(2651.4004317442873)

In [7]:
theoretical_max = eligible_df.iloc[test_idx].sort_values('missed_clicks_april', ascending=False).head(50)['missed_clicks_april'].sum()
print(f"Theoretical max (perfect oracle): {theoretical_max:,.0f} clicks")

Theoretical max (perfect oracle): 2,938 clicks


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Claim (from Week 5):**
"In this experiment, the Machine Learning models did not beat the simple Baseline heuristic on the metric that ultimately matters: raw business value (2,131 captured clicks for Baseline vs 1,274 for LR and 1,698 for RF). Because the problem is fundamentally algebraic (CTR Gap × Volume), the Baseline heuristic remains the recommended production system — it is more performant, perfectly interpretable, and requires zero ML infrastructure. However, the upward trend in value captured from Linear Regression to Random Forest is a real, promising signal, proving that more flexible models are worth investigating in future iterations."

**The Overclaim Audit:**
1. **"Proving that more flexible models are worth investigating"**: Overclaim. A two-point trend (LR -> RF) on a single test set doesn't *prove* anything; it only provides a directional hint.
2. **"A real, promising signal"**: Overclaim. Our Week 6 audit revealed this signal is fragile. The model completely crashes without volume features (proving it leans heavily on raw size, not just SEO nuance), and the entire evaluation rests on a single grouped split dominated by whale clients. 

**Rewritten Claim (Safe Language):**
"In this experiment, we **observed** that the Machine Learning models did not beat the simple Baseline heuristic on **measured** business value (2,131 captured clicks for Baseline vs. 1,274 for Linear Regression and 1,698 for Random Forest). Because the problem is fundamentally algebraic, the Baseline heuristic remains the recommended **decision-support** system—it is interpretable and requires no ML infrastructure. 

While we **measured** an upward trend in value captured from Linear Regression to Random Forest, this provides only a **directional** signal that more flexible models might be worth investigating. Furthermore, our Week 6 audit revealed important limitations: this performance was measured on a single grouped split (making absolute click numbers highly sensitive to which clients were assigned to the test set), and sensitivity testing showed the model relies almost entirely on raw volume features to achieve these numbers. Future work must validate across multiple client splits to confirm this directional trend."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.